# Comparative Sentiment Analysis - Major Tech Tickers

This notebook compares sentiment analysis results across the 5 major tech tickers to identify patterns and relative performance.

## Tickers Covered:
- **AAPL**: Apple Inc. (Consumer Electronics/Tech)
- **AMZN**: Amazon.com Inc. (E-commerce/Cloud)
- **GOOG**: Alphabet Inc. (Search/Advertising/AI)
- **META**: Meta Platforms Inc. (Social Media/Metaverse)
- **NVDA**: NVIDIA Corporation (AI/Semiconductors)

## Analysis Focus:
- Cross-ticker sentiment comparison
- Tech sector sentiment patterns
- Relative sentiment strength
- Time series comparison
- Statistical significance testing

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Load sentiment analysis results for all major tickers
print("Loading sentiment analysis results for major tech tickers...")

# Define major tickers to analyze
major_tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']
ticker_data = {}

# Load data for each ticker
for ticker in major_tickers:
    try:
        file_path = f'../../../data/sentiment_analysis_{ticker}.csv'
        data = pd.read_csv(file_path)
        ticker_data[ticker] = data
        print(f"Loaded {ticker}: {len(data)} headlines")
    except FileNotFoundError:
        print(f"Warning: Could not find data for {ticker}, loading from raw data...")
        # Fallback: load from raw data if individual files don't exist
        raw_data = pd.read_csv('../../../data/newsData/raw_analyst_ratings.csv')
        if ticker == 'META':
            # Handle META/FB transition
            ticker_filtered = raw_data[(raw_data['stock'] == ticker) | (raw_data['stock'] == 'FB')].copy()
            ticker_filtered['stock'] = ticker
        else:
            ticker_filtered = raw_data[raw_data['stock'] == ticker].copy()
        
        if len(ticker_filtered) > 0:
            print(f"Using raw data for {ticker}: {len(ticker_filtered)} headlines")
            ticker_data[ticker] = ticker_filtered
        else:
            print(f"No data found for {ticker}")

print(f"\nSuccessfully loaded data for {len(ticker_data)} major tickers")

In [ ]:
# Create comparative summary statistics
print("=== MAJOR TICKERS COMPARATIVE SENTIMENT SUMMARY ===")

summary_data = []

for ticker, data in ticker_data.items():
    # Check if sentiment scores exist, if not calculate them
    if 'textblob_polarity' not in data.columns:
        from textblob import TextBlob
        import nltk
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        
        # Download VADER if needed
        try:
            nltk.data.find('sentiment/vader_lexicon.zip')
        except LookupError:
            nltk.download('vader_lexicon')
        
        vader = SentimentIntensityAnalyzer()
        
        # Calculate sentiment scores
        data['textblob_polarity'] = data['headline'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
        data['textblob_subjectivity'] = data['headline'].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)
        data['vader_compound'] = data['headline'].apply(lambda x: vader.polarity_scores(str(x))['compound'])
    
    # Calculate statistics
    tb_mean = data['textblob_polarity'].mean()
    tb_std = data['textblob_polarity'].std()
    vader_mean = data['vader_compound'].mean()
    vader_std = data['vader_compound'].std()
    
    # Calculate sentiment categories
    def categorize_sentiment(score, tool='textblob'):
        if tool == 'textblob':
            if score > 0.1:
                return 'Positive'
            elif score < -0.1:
                return 'Negative'
            else:
                return 'Neutral'
        else:  # VADER
            if score >= 0.05:
                return 'Positive'
            elif score <= -0.05:
                return 'Negative'
            else:
                return 'Neutral'
    
    data['textblob_category'] = data['textblob_polarity'].apply(lambda x: categorize_sentiment(x, 'textblob'))
    data['vader_category'] = data['vader_compound'].apply(lambda x: categorize_sentiment(x, 'vader'))
    
    tb_positive = (data['textblob_category'] == 'Positive').mean()
    tb_negative = (data['textblob_category'] == 'Negative').mean()
    vader_positive = (data['vader_category'] == 'Positive').mean()
    vader_negative = (data['vader_category'] == 'Negative').mean()
    
    summary_data.append({
        'Ticker': ticker,
        'Headlines': len(data),
        'TB_Mean': tb_mean,
        'TB_Std': tb_std,
        'VADER_Mean': vader_mean,
        'VADER_Std': vader_std,
        'TB_Positive': tb_positive,
        'TB_Negative': tb_negative,
        'VADER_Positive': vader_positive,
        'VADER_Negative': vader_negative
    })

# Create summary dataframe
summary_df = pd.DataFrame(summary_data)
print("\nComparative Summary:")
print(summary_df.round(4))

# Sort by TextBlob mean sentiment
print("\nMajor Tickers ranked by TextBlob sentiment (most positive first):")
print(summary_df.sort_values('TB_Mean', ascending=False)[['Ticker', 'TB_Mean', 'VADER_Mean']].round(4))

print("\nMajor Tickers ranked by VADER sentiment (most positive first):")
print(summary_df.sort_values('VADER_Mean', ascending=False)[['Ticker', 'VADER_Mean', 'TB_Mean']].round(4))

In [ ]:
# Create comprehensive comparative visualizations
fig, axes = plt.subplots(3, 2, figsize=(16, 20))
fig.suptitle('Major Tech Tickers - Comparative Sentiment Analysis', fontsize=16, fontweight='bold')

# 1. Mean sentiment comparison
x = np.arange(len(summary_df))
width = 0.35

axes[0, 0].bar(x - width/2, summary_df['TB_Mean'], width, label='TextBlob', color='blue', alpha=0.7)
axes[0, 0].bar(x + width/2, summary_df['VADER_Mean'], width, label='VADER', color='orange', alpha=0.7)
axes[0, 0].set_title('Mean Sentiment Score Comparison')
axes[0, 0].set_xlabel('Ticker')
axes[0, 0].set_ylabel('Mean Sentiment Score')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(summary_df['Ticker'])
axes[0, 0].legend()
axes[0, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].grid(True, alpha=0.3)

# 2. Sentiment volatility comparison
axes[0, 1].bar(x - width/2, summary_df['TB_Std'], width, label='TextBlob', color='blue', alpha=0.7)
axes[0, 1].bar(x + width/2, summary_df['VADER_Std'], width, label='VADER', color='orange', alpha=0.7)
axes[0, 1].set_title('Sentiment Volatility (Std Dev) Comparison')
axes[0, 1].set_xlabel('Ticker')
axes[0, 1].set_ylabel('Standard Deviation')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(summary_df['Ticker'])
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Positive sentiment percentage comparison
axes[1, 0].bar(x - width/2, summary_df['TB_Positive'], width, label='TextBlob', color='green', alpha=0.7)
axes[1, 0].bar(x + width/2, summary_df['VADER_Positive'], width, label='VADER', color='darkgreen', alpha=0.7)
axes[1, 0].set_title('Positive Sentiment Percentage Comparison')
axes[1, 0].set_xlabel('Ticker')
axes[1, 0].set_ylabel('Positive Sentiment %')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(summary_df['Ticker'])
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Negative sentiment percentage comparison
axes[1, 1].bar(x - width/2, summary_df['TB_Negative'], width, label='TextBlob', color='red', alpha=0.7)
axes[1, 1].bar(x + width/2, summary_df['VADER_Negative'], width, label='VADER', color='darkred', alpha=0.7)
axes[1, 1].set_title('Negative Sentiment Percentage Comparison')
axes[1, 1].set_xlabel('Ticker')
axes[1, 1].set_ylabel('Negative Sentiment %')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(summary_df['Ticker'])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 5. News volume comparison
axes[2, 0].bar(summary_df['Ticker'], summary_df['Headlines'], color='purple', alpha=0.7)
axes[2, 0].set_title('News Volume Comparison')
axes[2, 0].set_xlabel('Ticker')
axes[2, 0].set_ylabel('Number of Headlines')
axes[2, 0].tick_params(axis='x', rotation=45)
axes[2, 0].grid(True, alpha=0.3)

# 6. Sentiment correlation heatmap
correlation_data = []
for ticker in summary_df['Ticker']:
    correlation_data.append([summary_df[summary_df['Ticker'] == ticker]['TB_Mean'].iloc[0],
                           summary_df[summary_df['Ticker'] == ticker]['VADER_Mean'].iloc[0]])

correlation_matrix = np.corrcoef(correlation_data)
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            xticklabels=summary_df['Ticker'], yticklabels=summary_df['Ticker'],
            ax=axes[2, 1], fmt='.3f')
axes[2, 1].set_title('Sentiment Correlation Matrix\n(TextBlob vs VADER)')

plt.tight_layout()
plt.show()

In [ ]:
# Sub-sector analysis
print("=== TECH SUB-SECTOR SENTIMENT ANALYSIS ===")

# Define sub-sectors for each ticker
subsector_mapping = {
    'AAPL': 'Consumer Electronics',
    'AMZN': 'E-commerce & Cloud',
    'GOOG': 'Search & AI',
    'META': 'Social Media',
    'NVDA': 'Semiconductors & AI'
}

# Add sub-sector information to summary
summary_df['SubSector'] = summary_df['Ticker'].map(subsector_mapping)

# Calculate sub-sector averages
subsector_summary = summary_df.groupby('SubSector').agg({
    'TB_Mean': 'mean',
    'VADER_Mean': 'mean',
    'Headlines': 'sum',
    'Ticker': 'count'
}).rename(columns={'Ticker': 'Count'})

print("\nSub-sector sentiment averages:")
print(subsector_summary.round(4))

# Create sub-sector comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Sub-sector mean sentiment comparison
subsectors = subsector_summary.index
x = np.arange(len(subsectors))
width = 0.35

ax1.bar(x - width/2, subsector_summary['TB_Mean'], width, label='TextBlob', color='blue', alpha=0.7)
ax1.bar(x + width/2, subsector_summary['VADER_Mean'], width, label='VADER', color='orange', alpha=0.7)
ax1.set_title('Tech Sub-Sector Sentiment Comparison')
ax1.set_xlabel('Sub-Sector')
ax1.set_ylabel('Mean Sentiment Score')
ax1.set_xticks(x)
ax1.set_xticklabels(subsectors, rotation=45, ha='right')
ax1.legend()
ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax1.grid(True, alpha=0.3)

# Sub-sector news volume
ax2.bar(subsectors, subsector_summary['Headlines'], color='purple', alpha=0.7)
ax2.set_title('Sub-Sector News Volume')
ax2.set_xlabel('Sub-Sector')
ax2.set_ylabel('Total Headlines')
ax2.tick_params(axis='x', rotation=45, ha='right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Time series comparison (if dates are available)
print("=== TIME SERIES SENTIMENT COMPARISON ===")

# Create time series data for each ticker
time_series_data = {}

for ticker, data in ticker_data.items():
    try:
        # Clean and convert dates
        data['date_clean'] = pd.to_datetime(data['date'], errors='coerce')
        time_data = data.dropna(subset=['date_clean']).copy()
        
        if len(time_data) > 0:
            time_data = time_data.sort_values('date_clean')
            time_data.set_index('date_clean', inplace=True)
            
            # Resample by month for smoother trends
            if 'textblob_polarity' in time_data.columns:
                monthly_sentiment = time_data['textblob_polarity'].resample('M').mean()
                time_series_data[ticker] = monthly_sentiment
                print(f"Created time series for {ticker}: {len(monthly_sentiment)} months")
            else:
                print(f"No sentiment data available for {ticker}")
    except Exception as e:
        print(f"Error processing time series for {ticker}: {e}")

# Plot time series comparison
if len(time_series_data) > 0:
    plt.figure(figsize=(16, 8))
    
    colors = {'AAPL': 'gray', 'AMZN': 'orange', 'GOOG': 'blue', 'META': 'royalblue', 'NVDA': 'green'}
    
    for ticker, series in time_series_data.items():
        plt.plot(series.index, series.values, marker='o', linewidth=2, 
                label=ticker, color=colors.get(ticker, 'black'), alpha=0.7)
    
    plt.title('Major Tech Tickers - Monthly Sentiment Trends Comparison')
    plt.xlabel('Date')
    plt.ylabel('Average TextBlob Polarity')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("No time series data available for comparison")

In [ ]:
# Statistical significance testing
print("=== STATISTICAL SIGNIFICANCE TESTING ===")

# Perform pairwise sentiment comparisons
tickers_with_data = [ticker for ticker in major_tickers if ticker in ticker_data and 'textblob_polarity' in ticker_data[ticker].columns]

if len(tickers_with_data) >= 2:
    print("\nPairwise TextBlob sentiment comparisons (t-test):")
    print("=" * 80)
    
    significant_results = []
    
    for i, ticker1 in enumerate(tickers_with_data):
        for ticker2 in tickers_with_data[i+1:]:
            data1 = ticker_data[ticker1]['textblob_polarity'].dropna()
            data2 = ticker_data[ticker2]['textblob_polarity'].dropna()
            
            if len(data1) > 0 and len(data2) > 0:
                t_stat, p_value = stats.ttest_ind(data1, data2)
                mean_diff = data1.mean() - data2.mean()
                
                significance = "Significant" if p_value < 0.05 else "Not significant"
                print(f"{ticker1} vs {ticker2}:")
                print(f"  Mean difference: {mean_diff:.4f}")
                print(f"  T-statistic: {t_stat:.4f}")
                print(f"  P-value: {p_value:.4f} ({significance})")
                print()
                
                if p_value < 0.05:
                    significant_results.append((ticker1, ticker2, mean_diff, p_value))

# ANOVA test for overall differences
if len(tickers_with_data) >= 3:
    print("\nANOVA test for overall sentiment differences:")
    print("=" * 60)
    
    sentiment_groups = []
    for ticker in tickers_with_data:
        data = ticker_data[ticker]['textblob_polarity'].dropna()
        if len(data) > 0:
            sentiment_groups.append(data)
    
    if len(sentiment_groups) >= 2:
        f_stat, p_value = stats.f_oneway(*sentiment_groups)
        significance = "Significant" if p_value < 0.05 else "Not significant"
        print(f"F-statistic: {f_stat:.4f}")
        print(f"P-value: {p_value:.4f} ({significance})")
        
        if p_value < 0.05:
            print("\nConclusion: There are statistically significant differences in sentiment between major tech tickers.")
            print(f"\nSignificant pairwise comparisons ({len(significant_results)}):")
            for ticker1, ticker2, mean_diff, p_val in significant_results:
                direction = "higher" if mean_diff > 0 else "lower"
                print(f"  {ticker1} sentiment {direction} than {ticker2} (p={p_val:.4f})")
        else:
            print("\nConclusion: No statistically significant differences found between major tech ticker sentiments.")

In [ ]:
# Investment insights and recommendations
print("=== MAJOR TECH TICKERS - INVESTMENT INSIGHTS ===")

# Calculate sentiment rankings
tb_ranking = summary_df.sort_values('TB_Mean', ascending=False)
vader_ranking = summary_df.sort_values('VADER_Mean', ascending=False)
volatility_ranking = summary_df.sort_values('TB_Std', ascending=False)
volume_ranking = summary_df.sort_values('Headlines', ascending=False)

print("\n1. SENTIMENT LEADERS:")
print(f"   TextBlob: {tb_ranking.iloc[0]['Ticker']} ({tb_ranking.iloc[0]['TB_Mean']:.4f})")
print(f"   VADER: {vader_ranking.iloc[0]['Ticker']} ({vader_ranking.iloc[0]['VADER_Mean']:.4f})")

print("\n2. MOST STABLE SENTIMENT (Lowest Volatility):")
most_stable = volatility_ranking.iloc[-1]
print(f"   {most_stable['Ticker']} (std dev: {most_stable['TB_Std']:.4f})")

print("\n3. HIGHEST NEWS COVERAGE:")
highest_volume = volume_ranking.iloc[-1]
print(f"   {highest_volume['Ticker']} ({highest_volume['Headlines']:,} headlines)")

print("\n4. SUB-SECTOR LEADERS:")
for subsector in subsector_summary.index:
    subsector_data = summary_df[summary_df['SubSector'] == subsector]
    if len(subsector_data) > 0:
        leader = subsector_data.loc[subsector_data['TB_Mean'].idxmax()]
        print(f"   {subsector}: {leader['Ticker']} ({leader['TB_Mean']:.4f})")

print("\n5. RELATIVE STRENGTH ANALYSIS:")
overall_mean = summary_df['TB_Mean'].mean()
above_average = summary_df[summary_df['TB_Mean'] > overall_mean]
print(f"   Tickers above average sentiment: {', '.join(above_average['Ticker'].tolist())}")
print(f"   Overall average sentiment: {overall_mean:.4f}")

print("\n6. RISK/REWARD PROFILE:")
summary_df['Risk_Reward'] = summary_df['TB_Mean'] / summary_df['TB_Std']
best_risk_reward = summary_df.loc[summary_df['Risk_Reward'].idxmax()]
print(f"   Best risk-adjusted sentiment: {best_risk_reward['Ticker']} (ratio: {best_risk_reward['Risk_Reward']:.4f})")

print("\n=== INVESTMENT RECOMMENDATIONS ===")
print("\nBased on sentiment analysis:")
print(f"1. CONSIDER {tb_ranking.iloc[0]['Ticker']} - Highest positive sentiment")
print(f"2. MONITOR {most_stable['Ticker']} - Most stable sentiment profile")
print(f"3. WATCH {highest_volume['Ticker']} - High news coverage indicates market attention")
print(f"4. DIVERSIFY across sub-sectors for balanced tech exposure")
print(f"5. USE sentiment trends as confirmation for fundamental analysis")

In [ ]:
# Save comparative results
print("=== SAVING COMPARATIVE RESULTS ===")

# Save summary dataframe
summary_file = '../../../data/major_tickers_sentiment_summary.csv'
summary_df.to_csv(summary_file, index=False)
print(f"Major tickers summary saved to: {summary_file}")

# Save sub-sector summary
subsector_file = '../../../data/tech_subsectors_sentiment_summary.csv'
subsector_summary.to_csv(subsector_file)
print(f"Sub-sector summary saved to: {subsector_file}")

# Create detailed insights report
insights_file = '../../../data/major_tickers_insights.txt'
with open(insights_file, 'w') as f:
    f.write("MAJOR TECH TICKERS SENTIMENT ANALYSIS INSIGHTS\n")
    f.write("=" * 50 + "\n\n")
    
    f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d')}\n\n")
    f.write(f"Tickers Analyzed: {', '.join(major_tickers)}\n")
    f.write(f"Total Headlines: {summary_df['Headlines'].sum():,}\n\n")
    
    f.write("KEY FINDINGS:\n")
    f.write(f"- Highest Sentiment: {tb_ranking.iloc[0]['Ticker']} ({tb_ranking.iloc[0]['TB_Mean']:.4f})\n")
    f.write(f"- Most Stable: {most_stable['Ticker']} (std: {most_stable['TB_Std']:.4f})\n")
    f.write(f"- Highest Coverage: {highest_volume['Ticker']} ({highest_volume['Headlines']:,})\n")
    f.write(f"- Best Risk-Adjusted: {best_risk_reward['Ticker']} (ratio: {best_risk_reward['Risk_Reward']:.4f})\n\n")
    
    f.write("SUB-SECTOR PERFORMANCE:\n")
    for subsector in subsector_summary.index:
        subsector_data = summary_df[summary_df['SubSector'] == subsector]
        if len(subsector_data) > 0:
            leader = subsector_data.loc[subsector_data['TB_Mean'].idxmax()]
            f.write(f"- {subsector}: {leader['Ticker']} ({leader['TB_Mean']:.4f})\n")
    
    f.write(f"\nSTATISTICAL SIGNIFICANCE: {len(significant_results)} significant pairwise differences found\n")

print(f"Detailed insights saved to: {insights_file}")

print(f"\n=== ANALYSIS COMPLETE ===")
print(f"Analyzed {len(tickers_with_data)} major tech tickers across {len(subsector_summary)} sub-sectors")
print(f"Total headlines processed: {summary_df['Headlines'].sum():,}")
print(f"Sentiment range: {summary_df['TB_Mean'].min():.4f} to {summary_df['TB_Mean'].max():.4f}")

## Major Tech Tickers - Comparative Analysis Summary

### Key Findings:

#### **Sentiment Leaders:**
- [Most positive ticker by TextBlob]
- [Most positive ticker by VADER]
- [Best risk-adjusted sentiment]

#### **Sub-Sector Performance:**
- Consumer Electronics: [AAPL performance]
- E-commerce & Cloud: [AMZN performance]
- Search & AI: [GOOG performance]
- Social Media: [META performance]
- Semiconductors & AI: [NVDA performance]

#### **Statistical Insights:**
- [Significant differences between tickers]
- [ANOVA test results]
- [Tool agreement patterns]

### Investment Implications:

#### **Relative Sentiment Analysis:**
- Compare individual ticker sentiment against tech sector averages
- Identify sentiment leaders and laggards within sub-sectors
- Monitor sentiment convergence/divergence patterns

#### **Sub-Sector Rotation Signals:**
- Use sub-sector sentiment trends for rotation strategies
- Identify emerging tech sub-sector strength/weakness
- Monitor cross-sub-sector sentiment spreads

#### **Risk Management:**
- High volatility tickers may present higher risk/reward
- Sentiment divergence between tools may indicate uncertainty
- News volume can signal upcoming volatility and market attention

### Recommendations:

1. **Regular Monitoring**: Track sentiment trends weekly for early signals
2. **Cross-Validation**: Use both TextBlob and VADER for robust analysis
3. **Sub-Sector Context**: Always consider ticker sentiment within sub-sector context
4. **Volume Analysis**: Combine sentiment with news volume for better signals
5. **Statistical Validation**: Use statistical tests to confirm significant differences
6. **Risk-Adjusted Analysis**: Consider sentiment volatility in investment decisions

### Trading Strategies:

- **Momentum**: Follow sentiment leaders with positive trends
- **Mean Reversion**: Consider extreme sentiment positions for reversal opportunities
- **Pairs Trading**: Exploit sentiment differences between correlated tickers
- **Sector Rotation**: Rotate capital between sub-sectors based on relative sentiment